In [5]:
import pandas as pd

# 1. Carregar o dataset do Kaggle
df_daigt = pd.read_csv('data/train_v4_drcat_01.csv')

print(df_daigt['model'].value_counts())

model
human      27373
llama      15796
mistral    13439
falcon      4536
gpt         4161
davinci     2099
claude      2000
palm        1733
babbage      698
curie        696
ada          692
cohere       350
Name: count, dtype: int64


In [7]:
# Criar a coluna 'label_oficial' vazia
df_daigt['label_oficial'] = None

# 2. Mapeamento
# Humanos:
df_daigt.loc[df_daigt['model'] == 'human', 'label_oficial'] = 0

# AIs: Agrupadas com base no print
openai_models = ['gpt', 'davinci', 'babbage', 'curie', 'ada']
google_models = ['palm']
meta_models = ['llama']
anthropic_models = ['claude']

# Aplicar as regras
df_daigt.loc[df_daigt['model'].isin(openai_models), 'label_oficial'] = 1
df_daigt.loc[df_daigt['model'].isin(google_models), 'label_oficial'] = 2
df_daigt.loc[df_daigt['model'].isin(meta_models), 'label_oficial'] = 3
df_daigt.loc[df_daigt['model'].isin(anthropic_models), 'label_oficial'] = 4

# 3. Limpar o lixo (remover mistral, falcon, cohere, etc.)
df_limpo = df_daigt.dropna(subset=['label_oficial']).copy()
df_limpo['label_oficial'] = df_limpo['label_oficial'].astype(int)

# 4. Balancear o Dataset
# O número mínimo vai ser 1733 (do 'palm'). O pandas vai garantir que extrai exatamente isso de todas as outras
QTD_POR_CLASSE = df_limpo['label_oficial'].value_counts().min()
print(f"A balancear o dataset para ter exatamente {QTD_POR_CLASSE} textos de cada categoria...")

df_balanceado = df_limpo.groupby('label_oficial').sample(n=QTD_POR_CLASSE, random_state=42)

# 5. Guardar o ficheiro final 
df_final = df_balanceado[['text', 'label_oficial']].rename(columns={'label_oficial': 'label'})

# Misturar as linhas para não ficarem todas juntas
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

df_final.to_csv('data/dataset.csv', index=False)

print(df_final['label'].value_counts().sort_index())
print("\nLegenda: 0=Humano, 1=OpenAI, 2=Google, 3=Meta, 4=Anthropic")

A balancear o dataset para ter exatamente 1733 textos de cada categoria...
label
0    1733
1    1733
2    1733
3    1733
4    1733
Name: count, dtype: int64

Legenda: 0=Humano, 1=OpenAI, 2=Google, 3=Meta, 4=Anthropic
